# Neural Machine Translation


## Table of Contents

* [Project Setup and Required Libraries](#project-setup-and-required-libraries)
* [1. Translating Human-Readable Dates into Machine-Readable Dates](#1-translating-human-readable-dates-into-machine-readable-dates)

  * [1.1 Dataset Overview](#11-dataset-overview)
  * [1.2 Data Preprocessing](#12-data-preprocessing)
* [2. Building a Neural Machine Translation Model with Attention](#2-building-a-neural-machine-translation-model-with-attention)

  * [2.1 Understanding the Attention Mechanism](#21-understanding-the-attention-mechanism)
  * [2.2 Implementing One-Step Attention](#22-implementing-one-step-attention)
  * [2.3 Building the Encoder-Decoder Model](#23-building-the-encoder-decoder-model)
  * [2.4 Compiling and Training the Model](#24-compiling-and-training-the-model)
* [3. Visualizing Model Attention](#3-visualizing-model-attention)

  * [3.1 Extracting Attention Weights from the Network](#31-extracting-attention-weights-from-the-network)
  * [3.2 Interpreting Attention Patterns](#32-interpreting-attention-patterns)


<a name='0'></a>
## Packages

In [ ]:
### v3.1

In [2]:
from tensorflow.keras.layers import Bidirectional, Concatenate, Permute, Dot, Input, LSTM, Multiply
from tensorflow.keras.layers import RepeatVector, Dense, Activation, Lambda
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import load_model, Model
import tensorflow.keras.backend as K
import tensorflow as tf
import numpy as np

from faker import Faker
import random
from tqdm import tqdm
from babel.dates import format_date
from nmt_utils import *
import matplotlib.pyplot as plt
%matplotlib inline

<a name='1'></a>

## 1. Translating Human-Readable Dates into Machine-Readable Dates

In this project, I implemented a simplified Neural Machine Translation task using human-readable dates.

The model takes dates written in different formats, such as:

* `the 29th of August 1958`
* `03/30/1968`
* `24 JUNE 1987`

and translates them into the standard machine-readable format:

* `1958-08-29`
* `1968-03-30`
* `1987-06-24`

This task demonstrates the core idea behind sequence-to-sequence modeling with attention while keeping the dataset simple and easy to interpret.


<a name='1-1'></a>

### 1.1 Dataset

I used a dataset of **10,000 human-readable dates** paired with their standardized machine-readable format.

Each example maps an input date, written in different natural formats, to the target `YYYY-MM-DD` format. This allows the model to learn date translation as a simplified Neural Machine Translation task.


In [3]:
m = 10000
dataset, human_vocab, machine_vocab, inv_machine_vocab = load_dataset(m)

100%|██████████| 10000/10000 [00:00<00:00, 22552.01it/s]


In [4]:
dataset[:10]

[('9 may 1998', '1998-05-09'),
 ('10.11.19', '2019-11-10'),
 ('9/10/70', '1970-09-10'),
 ('friday october 24 2025', '2025-10-24'),
 ('monday august 19 2024', '2024-08-19'),
 ('saturday april 28 1990', '1990-04-28'),
 ('thursday january 26 1995', '1995-01-26'),
 ('07 mar 1983', '1983-03-07'),
 ('22 may 1988', '1988-05-22'),
 ('tuesday july 8 2008', '2008-07-08')]

The dataset has been loaded with the following components:

* `dataset`: pairs of human-readable and machine-readable dates
* `human_vocab`: character-to-index mapping for input dates
* `machine_vocab`: character-to-index mapping for output dates
* `inv_machine_vocab`: index-to-character mapping for decoding predictions

Next, I preprocess the raw text by converting each character into its corresponding index.

I set:

* `Tx = 30`, the maximum input sequence length
* `Ty = 10`, the output sequence length for the `YYYY-MM-DD` format


In [5]:
Tx = 30
Ty = 10
X, Y, Xoh, Yoh = preprocess_data(dataset, human_vocab, machine_vocab, Tx, Ty)

print("X.shape:", X.shape)
print("Y.shape:", Y.shape)
print("Xoh.shape:", Xoh.shape)
print("Yoh.shape:", Yoh.shape)

X.shape: (10000, 30)
Y.shape: (10000, 10)
Xoh.shape: (10000, 30, 37)
Yoh.shape: (10000, 10, 11)


After preprocessing, the data is represented in four formats:

* `X`: indexed human-readable input dates, padded to length `Tx`
* `Y`: indexed machine-readable target dates with length `Ty`
* `Xoh`: one-hot encoded version of `X`
* `Yoh`: one-hot encoded version of `Y`

The resulting shapes are:

* `X.shape = (m, Tx)`
* `Y.shape = (m, Ty)`
* `Xoh.shape = (m, Tx, len(human_vocab))`
* `Yoh.shape = (m, Ty, len(machine_vocab))`

Here, `m` represents the number of training examples.


Here, I inspect a few preprocessed examples to verify how the original input dates and target dates were converted into indexed and one-hot encoded representations.

In [6]:
index = 0
print("Source date:", dataset[index][0])
print("Target date:", dataset[index][1])
print()
print("Source after preprocessing (indices):", X[index])
print("Target after preprocessing (indices):", Y[index])
print()
print("Source after preprocessing (one-hot):", Xoh[index])
print("Target after preprocessing (one-hot):", Yoh[index])

Source date: 9 may 1998
Target date: 1998-05-09

Source after preprocessing (indices): [12  0 24 13 34  0  4 12 12 11 36 36 36 36 36 36 36 36 36 36 36 36 36 36
 36 36 36 36 36 36]
Target after preprocessing (indices): [ 2 10 10  9  0  1  6  0  1 10]

Source after preprocessing (one-hot): [[0. 0. 0. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 0. 1.]]
Target after preprocessing (one-hot): [[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]]


<a name='2'></a>

## 2. Neural Machine Translation with Attention

In this section, I build a Neural Machine Translation model using an attention mechanism.

Attention helps the model focus on the most relevant parts of the input sequence while generating each character of the output date.

<a name='2-1'></a>

### 2.1 Attention Mechanism

The attention mechanism computes a context vector for each output timestep. This allows the model to decide which parts of the input date are most important when predicting each character in the `YYYY-MM-DD` output.

<table>
<td> 
<img src="images/attn_model.png" style="width:500;height:500px;"> <br>
</td> 
<td> 
<img src="images/attn_mechanism.png" style="width:500;height:500px;"> <br>
</td> 
</table>

<caption><center><b>Figure 1:</b> Neural Machine Translation with Attention</center></caption>

The model uses two LSTM components:

* **Pre-attention Bi-LSTM**: processes the input sequence across `Tx` timesteps and creates contextual representations of the human-readable date.
* **Post-attention LSTM**: generates the output sequence across `Ty` timesteps using the context vectors produced by the attention mechanism.

At each output step, the post-attention LSTM updates its hidden state `s<t>` and cell state `c<t>` to predict the next character in the machine-readable date.


In this implementation, the post-attention sequence model uses an **LSTM** instead of a basic RNN.

This means the model maintains two internal states at each timestep:

* the hidden state `s<t>`
* the cell state `c<t>`

Together, these states help the model retain information while generating the output date sequence.


In this model, each output timestep does not use the previous predicted character as input.

Instead, the post-attention LSTM uses its hidden state `s<t>` and cell state `c<t>` to generate each character of the output date.

This works well for the `YYYY-MM-DD` format because the task is more structured than open-ended language generation.


The pre-attention layer uses a **Bidirectional LSTM**, so each input timestep is represented using information from both directions.

The forward and backward hidden states are concatenated as:

`a<t> = [forward_a<t>, backward_a<t>]`

This gives the attention mechanism a richer representation of each character in the input date.


#### Computing "energies" $e^{\langle t, t' \rangle}$ as a function of $s^{\langle t-1 \rangle}$ and $a^{\langle t' \rangle}$

At each output timestep, the attention layer computes an energy score `e<t,t'>`.

This score measures how relevant each input representation `a<t'>` is for generating the current output character.

The energy scores are then converted into attention weights, which determine how much focus the model should place on each part of the input sequence.


To compute attention, I repeat the previous hidden state `s<t-1>` across the input sequence length `Tx`.

Then, I concatenate it with each pre-attention hidden state `a<t'>` and pass the result through a Dense layer to produce energy scores.

A softmax layer converts these energy scores into attention weights, showing how much each input character contributes to the current output prediction.


#### Implementation Details
   
To compute attention, I repeat the previous hidden state `s<t-1>` across the input sequence length `Tx`.

Then, I concatenate it with each pre-attention hidden state `a<t'>` and pass the result through a Dense layer to produce energy scores.

A softmax layer converts these energy scores into attention weights, showing how much each input character contributes to the current output prediction.

<a name='ex-1'></a>

### 2.2 One-Step Attention

I implemented the `one_step_attention()` function to compute the context vector for one output timestep.

This function:

* repeats the previous hidden state across the input sequence
* concatenates it with the pre-attention hidden states
* computes attention energy scores
* applies softmax to obtain attention weights
* produces the context vector used by the decoder

The same attention layers are reused across all `Ty` output steps so the model shares weights consistently during sequence generation.


In [ ]:

# Defined shared layers as global variables
repeator = RepeatVector(Tx)
concatenator = Concatenate(axis=-1)
densor1 = Dense(10, activation = "tanh")
densor2 = Dense(1, activation = "relu")
activator = Activation(softmax, name='attention_weights') # We are using a custom softmax(axis = 1) loaded in this notebook
dotor = Dot(axes = 1)

In [ ]:
```python
def one_step_attention(a, s_prev):
    """
    Computes one step of the attention mechanism.

    The function calculates attention weights over the input sequence and uses
    them to produce a context vector for the decoder LSTM.

    Parameters
    ----------
    a : tensor
        Hidden states from the pre-attention Bi-LSTM with shape (m, Tx, 2*n_a).
    s_prev : tensor
        Previous hidden state from the post-attention LSTM with shape (m, n_s).

    Returns
    -------
    context : tensor
        Context vector passed to the post-attention LSTM.
    """

    # Repeat the previous decoder hidden state across all input timesteps
    s_prev = repeator(s_prev)

    # Combine encoder hidden states with the repeated decoder hidden state
    concat = concatenator([a, s_prev])

    # Compute intermediate attention scores
    e = densor1(concat)

    # Compute energy values for each input timestep
    energies = densor2(e)

    # Convert energy values into attention weights
    alphas = activator(energies)

    # Compute the context vector as a weighted sum of encoder hidden states
    context = dotor([alphas, a])

    return context
```


In [ ]:

# UNIT TEST
def one_step_attention_test(target):
    np.random.seed(10)
    tf.random.set_seed(10)

    m = 10
    Tx = 30
    n_a = 32
    n_s = 64
    
    a = np.random.uniform(1, 0, (m, Tx, 2 * n_a)).astype(np.float32)
    s_prev =np.random.uniform(1, 0, (m, n_s)).astype(np.float32) * 1
    context = target(a, s_prev)
    
    expected_output = np.load('expected_output_ex1.npy')
    
    assert type(context) == tf.python.framework.ops.EagerTensor, "Unexpected type. It should be a Tensor"
    assert tuple(context.shape) == (m, 1, n_s), "Unexpected output shape"
    assert np.all(context.numpy() == expected_output), "Unexpected values in the result"
    
    print("\033[92mAll tests passed!")
    
one_step_attention_test(one_step_attention)

All tests passed!


<a name='ex-2'></a>

### 2.3 Encoder-Decoder Model

I implemented the full Neural Machine Translation model using an encoder-decoder architecture with attention.

The model first passes the input sequence through a Bidirectional LSTM to generate contextual hidden states for each input character.

Then, for each output timestep, the model:

* computes a context vector using `one_step_attention()`
* passes the context vector into the post-attention LSTM
* applies a Dense layer with softmax activation
* predicts the next character in the `YYYY-MM-DD` output sequence

The attention and decoder layers are reused across all output timesteps to keep the model weights shared during sequence generation.


In [ ]:

n_a = 32 # number of units for the pre-attention, bi-directional LSTM's hidden state 'a'
n_s = 64 # number of units for the post-attention LSTM's hidden state "s"

this is the post attention LSTM cell.  
post_activation_LSTM_cell = LSTM(n_s, return_state = True) # Please do not modify this global variable.
output_layer = Dense(len(machine_vocab), activation=softmax)

The model generates the output sequence one character at a time across `Ty` timesteps.

First, the one-hot encoded input `X` is passed through a Bidirectional LSTM to produce hidden states for every input character.

Then, at each output timestep, the model:

* computes a context vector using attention
* updates the post-attention LSTM hidden and cell states
* applies a softmax output layer
* predicts the next character in the machine-readable date

Finally, I define the Keras model with three inputs: the encoded input sequence `X`, the initial hidden state `s0`, and the initial cell state `c0`.


In [ ]:


def modelf(Tx, Ty, n_a, n_s, human_vocab_size, machine_vocab_size=None):
    """
    Arguments:
    Tx -- length of the input sequence
    Ty -- length of the output sequence
    n_a -- hidden state size of the Bi-LSTM
    n_s -- hidden state size of the post-attention LSTM
    human_vocab_size -- size of the python dictionary "human_vocab"
    machine_vocab_size -- integer, optional, size of the python dictionary "machine_vocab"
                          This is not being used

    Returns:
    model -- Keras model instance
    """
    
    # Define the inputs of your model with a shape (Tx, human_vocab_size)
    # Define s0 (initial hidden state) and c0 (initial cell state)
    # for the decoder LSTM with shape (n_s,)
    X = Input(shape=(Tx, human_vocab_size))
    # initial hidden state
    s0 = Input(shape=(n_s,), name='s0')
    # initial cell state
    c0 = Input(shape=(n_s,), name='c0')
    # hidden state
    s = s0
    # cell state
    c = c0
    
    # Initialize empty list of outputs
    outputs = []
    
    
    # Step 1: Define your pre-attention Bi-LSTM. (≈ 1 line)
    a = Bidirectional(LSTM(n_a, return_sequences=True))(X)
    
    # Step 2: Iterate for Ty steps
    for t in range(Ty):
    
        # Step 2.A: Perform one step of the attention mechanism to get back the context vector at step t (≈ 1 line)
        context = one_step_attention(a, s)
        
        # Step 2.B: Apply the post-attention LSTM cell to the "context" vector. (≈ 1 line)
        # Don't forget to pass: initial_state = [hidden state, cell state] 
        # Remember: s = hidden state, c = cell state
        s, _, c = post_activation_LSTM_cell(context, initial_state=[s, c])
        
        # Step 2.C: Apply Dense layer to the hidden state output of the post-attention LSTM (≈ 1 line)
        out = output_layer(s)
        
        # Step 2.D: Append "out" to the "outputs" list (≈ 1 line)
        outputs.append(out)
    
    # Step 3: Create model instance taking three inputs and returning the list of outputs. (≈ 1 line)
    model = Model(inputs=[X, s0, c0], outputs=outputs)
    
    
    return model


In [ ]:
from test_utils import *

def modelf_test(target):
    Tx = 30
    n_a = 32
    n_s = 64
    len_human_vocab = 37
    
    
    model = target(Tx, Ty, n_a, n_s, len_human_vocab)
    
    print(summary(model))

    
    expected_summary = [['InputLayer', [(None, 30, 37)], 0],
                         ['InputLayer', [(None, 64)], 0],
                         ['Bidirectional', (None, 30, 64), 17920],
                         ['RepeatVector', (None, 30, 64), 0, 30],
                         ['Concatenate', (None, 30, 128), 0],
                         ['Dense', (None, 30, 10), 1290, 'tanh'],
                         ['Dense', (None, 30, 1), 11, 'relu'],
                         ['Activation', (None, 30, 1), 0],
                         ['Dot', (None, 1, 64), 0],
                         ['InputLayer', [(None, 64)], 0],
                         ['LSTM',[(None, 64), (None, 64), (None, 64)], 33024,[(None, 1, 64), (None, 64), (None, 64)],'tanh'],
                         ['Dense', (None, 11), 715, 'softmax']]

    assert len(model.outputs) == 10, f"Wrong output shape. Expected 10 != {len(model.outputs)}"

    comparator(summary(model), expected_summary)
    

modelf_test(modelf)

[['InputLayer', [(None, 30, 37)], 0], ['InputLayer', [(None, 64)], 0], ['Bidirectional', (None, 30, 64), 17920], ['RepeatVector', (None, 30, 64), 0, 30], ['Concatenate', (None, 30, 128), 0], ['Dense', (None, 30, 10), 1290, 'tanh'], ['Dense', (None, 30, 1), 11, 'relu'], ['Activation', (None, 30, 1), 0], ['Dot', (None, 1, 64), 0], ['InputLayer', [(None, 64)], 0], ['LSTM', [(None, 64), (None, 64), (None, 64)], 33024, [(None, 1, 64), (None, 64), (None, 64)], 'tanh'], ['Dense', (None, 11), 715, 'softmax']]
All tests passed!


Running the test in the next cell to check if the `modelf` function is correctly updating the (next) `hidden state` and `cell state`.

In [ ]:
### You cannot edit this cell

# UNIT TEST 2
def modelf_states_test(target):
    Tx = 30
    n_a = 32
    n_s = 64
    len_human_vocab = 37

    model = target(Tx, Ty, n_a, n_s, len_human_vocab)

    # Create test inputs
    X_test = np.random.rand(1, Tx, len_human_vocab)
    s0_test = np.zeros((1, n_s))
    c0_test = np.zeros((1, n_s))

    @tf.function(input_signature=[
        tf.TensorSpec(shape=[None, Tx, len_human_vocab], dtype=tf.float32),
        tf.TensorSpec(shape=[None, n_s], dtype=tf.float32),
        tf.TensorSpec(shape=[None, n_s], dtype=tf.float32)
    ])
    def predict_function(X, s0, c0):
        # Call the model directly with input tensors
        return model([X, s0, c0])  

    # Get the outputs of the model for the first five time steps
    outputs = predict_function(X_test, s0_test, c0_test)

    # Extract the hidden states (s) from the LSTM outputs for each time step
    hidden_states = [np.array(output) for output in outputs]

    # Check if consecutive hidden states are different
    for i in range(len(hidden_states) - 1):
        assert not np.allclose(hidden_states[i], hidden_states[i + 1]), (
            "Consecutive hidden states should be different.\n"
            "Check if the LSTM cell is using the correct previous states.\n"
            "Make sure you are using s and c, and NOT using s0 and c0 in Step 2.B."
        )

    print("\033[32mAll tests passed!\033[0m")

modelf_states_test(modelf)

All tests passed!


Running the following cell to create the model.

In [14]:
## You cannot edit this cell
model = modelf(Tx, Ty, n_a, n_s, len(human_vocab))

#### Troubleshooting Note
* If you are getting repeated errors after an initially incorrect implementation of "model", but believe that you have corrected the error, you may still see error messages when building your model.  
* A solution is to save and restart your kernel (or shutdown then restart your notebook), and re-run the cells.

Let's get a summary of the model to check if it matches the expected output.

In [15]:
model.summary()

Model: "functional_5"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_3 (InputLayer)            [(None, 30, 37)]     0                                            
__________________________________________________________________________________________________
s0 (InputLayer)                 [(None, 64)]         0                                            
__________________________________________________________________________________________________
bidirectional_2 (Bidirectional) (None, 30, 64)       17920       input_3[0][0]                    
__________________________________________________________________________________________________
repeat_vector (RepeatVector)    (None, 30, 64)       0           s0[0][0]                         
                                                                 lstm[20][0]           

**Expected Output**:

Here is the summary you should see
<table>
    <tr>
        <td>
            **Total params:**
        </td>
        <td>
         52,960
        </td>
    </tr>
        <tr>
        <td>
            **Trainable params:**
        </td>
        <td>
         52,960
        </td>
    </tr>
            <tr>
        <td>
            **Non-trainable params:**
        </td>
        <td>
         0
        </td>
    </tr>
                    <tr>
        <td>
            **bidirectional_1's output shape **
        </td>
        <td>
         (None, 30, 64)  
        </td>
    </tr>
    <tr>
        <td>
            **repeat_vector_1's output shape **
        </td>
        <td>
         (None, 30, 64) 
        </td>
    </tr>
                <tr>
        <td>
            **concatenate_1's output shape **
        </td>
        <td>
         (None, 30, 128) 
        </td>
    </tr>
            <tr>
        <td>
            **attention_weights's output shape **
        </td>
        <td>
         (None, 30, 1)  
        </td>
    </tr>
        <tr>
        <td>
            **dot_1's output shape **
        </td>
        <td>
         (None, 1, 64)
        </td>
    </tr>
           <tr>
        <td>
            **dense_3's output shape **
        </td>
        <td>
         (None, 11) 
        </td>
    </tr>
</table>


<a name='ex-3'></a>

### 2.4 Model Compilation

After defining the architecture, I compiled the model using categorical cross-entropy loss and the Adam optimizer.

The optimizer was configured with:

* learning rate: `0.005`
* `beta_1`: `0.9`
* `beta_2`: `0.999`
* decay: `0.01`

Accuracy was used as the evaluation metric to track character-level prediction performance during training.


In [ ]:
opt = Adam(lr=0.005, beta_1=0.9, beta_2=0.999, decay=0.01)
model.compile(loss='categorical_crossentropy', optimizer=opt, metrics=['accuracy'])


In [ ]:

# UNIT TESTS
assert opt.lr == 0.005, "Set the lr parameter to 0.005"
assert opt.beta_1 == 0.9, "Set the beta_1 parameter to 0.9"
assert opt.beta_2 == 0.999, "Set the beta_2 parameter to 0.999"
assert opt.decay == 0.01, "Set the decay parameter to 0.01"
assert model.loss == "categorical_crossentropy", "Wrong loss. Use 'categorical_crossentropy'"
assert model.optimizer == opt, "Use the optimizer that you have instantiated"
assert model.compiled_metrics._user_metrics[0] == 'accuracy', "set metrics to ['accuracy']"

print("\033[92mAll tests passed!")

All tests passed!


### Model Inputs and Training

To train the model, I used the one-hot encoded input data `Xoh`.

I also initialized the decoder hidden state `s0` and cell state `c0` with zeros before training.

Since the target date format has 10 characters, the model produces a list of 10 outputs, with each output representing one predicted character in the `YYYY-MM-DD` sequence.


In [18]:
### You cannot edit this cell
s0 = np.zeros((m, n_s))
c0 = np.zeros((m, n_s))
outputs = list(Yoh.swapaxes(0,1))

Let's now fit the model and run it for one epoch.

In [ ]:
model.fit([Xoh, s0, c0], outputs, epochs=1, batch_size=100)

100/100 [==============================] - 11s 111ms/step - loss: 17.1271 - dense_2_loss: 1.1394 - dense_2_1_loss: 1.0397 - dense_2_2_loss: 1.8401 - dense_2_3_loss: 2.6657 - dense_2_4_loss: 0.8434 - dense_2_5_loss: 1.3562 - dense_2_6_loss: 2.6967 - dense_2_7_loss: 1.1530 - dense_2_8_loss: 1.8017 - dense_2_9_loss: 2.5912 - dense_2_accuracy: 0.5394 - dense_2_1_accuracy: 0.6818 - dense_2_2_accuracy: 0.2739 - dense_2_3_accuracy: 0.0887 - dense_2_4_accuracy: 0.9052 - dense_2_5_accuracy: 0.2991 - dense_2_6_accuracy: 0.0357 - dense_2_7_accuracy: 0.8311 - dense_2_8_accuracy: 0.2176 - dense_2_9_accuracy: 0.1006


During training, the model reports the loss and accuracy for each of the 10 output positions.

Each accuracy value shows how well the model predicts a specific character in the `YYYY-MM-DD` output sequence.

<img src="images/table.png" style="width:700;height:200px;"> <br>

<caption><center><b>Example:</b> an accuracy of `0.89` at one output position means the model predicted that character correctly 89% of the time in the current batch.</center></caption>

To save training time, I loaded pre-trained weights for the model and used them to evaluate predictions.


In [ ]:
model.load_weights('models/model.h5')

The results on new examples.

In [ ]:

EXAMPLES = ['3 May 1979', '5 April 09', '21th of August 2016', 'Tue 10 Jul 2007', 'Saturday May 9 2018', 'March 3 2001', 'March 3rd 2001', '1 March 2001']
s00 = np.zeros((1, n_s))
c00 = np.zeros((1, n_s))
for example in EXAMPLES:
    source = string_to_int(example, Tx, human_vocab)
    #print(source)
    source = np.array(list(map(lambda x: to_categorical(x, num_classes=len(human_vocab)), source))).swapaxes(0,1)
    source = np.swapaxes(source, 0, 1)
    source = np.expand_dims(source, axis=0)
    prediction = model.predict([source, s00, c00])
    prediction = np.argmax(prediction, axis = -1)
    output = [inv_machine_vocab[int(i)] for i in prediction]
    print("source:", example)
    print("output:", ''.join(output),"\n")

source: 3 May 1979
output: 1979-05-33 

source: 5 April 09
output: 2009-04-05 

source: 21th of August 2016
output: 2016-08-20 

source: Tue 10 Jul 2007
output: 2007-07-10 

source: Saturday May 9 2018
output: 2018-05-09 

source: March 3 2001
output: 2001-03-03 

source: March 3rd 2001
output: 2001-03-03 

source: 1 March 2001
output: 2001-03-01 



I tested the trained model on different date examples to check how well it converts human-readable dates into the `YYYY-MM-DD` format.

In the next section, I visualize the attention weights to understand which parts of the input sequence the model focuses on while generating each output character.


<a name='3'></a>

## 3. Visualizing Attention

In this section, I visualize the attention weights learned by the model.

The attention map shows which parts of the input date the model focuses on when generating each character of the output date.

For example, when translating:

`Saturday 9 May 2018` → `2018-05-09`

the model learns to focus on the relevant parts of the input, such as the year, month, and day, while mostly ignoring unnecessary text like `Saturday`.

<img src="images/date_attention.png" style="width:600;height:300px;"> <br>

<caption><center><b>Figure 8:</b> Attention map for date translation</center></caption>


<a name='3-1'></a>

### 3.1 Extracting Attention Weights from the Network

To visualize attention, I passed an example date through the trained model and extracted the attention weights.

These weights show how strongly each output character attends to each character in the input sequence.

I first inspected the model summary to identify the layers where the attention values are computed.


In [22]:
model.summary()

Model: "functional_5"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_3 (InputLayer)            [(None, 30, 37)]     0                                            
__________________________________________________________________________________________________
s0 (InputLayer)                 [(None, 64)]         0                                            
__________________________________________________________________________________________________
bidirectional_2 (Bidirectional) (None, 30, 64)       17920       input_3[0][0]                    
__________________________________________________________________________________________________
repeat_vector (RepeatVector)    (None, 30, 64)       0           s0[0][0]                         
                                                                 lstm[20][0]           

From the model summary, I identified the `attention_weights` layer, which stores the attention values before the context vector is computed.

These attention weights have shape `(m, Tx, 1)` and show how much each input character contributes to a specific output timestep.

I used the `plot_attention_map()` function to extract and visualize these values as an attention heatmap.


In [ ]:
attention_map = plot_attention_map(model, human_vocab, inv_machine_vocab, "Tuesday 09 Oct 1993", num = 7, n_s = 64);

The attention plot helps interpret what the model focuses on while generating each output character.

For the date translation task, the model usually pays stronger attention to the input characters related to the year. The day and month are often easier to infer because the output format is fixed and highly structured.



#### Here's things to should remember

- Machine translation models can be used to map from one sequence to another. They are useful not just for translating human languages (like French->English) but also for tasks like date format translation. 
- An attention mechanism allows a network to focus on the most relevant parts of the input when producing a specific part of the output. 
- A network using an attention mechanism can translate from inputs of length $T_x$ to outputs of length $T_y$, where $T_x$ and $T_y$ can be different. 
- You can visualize attention weights $\alpha^{\langle t,t' \rangle}$ to see what the network is paying attention to while generating each output.

Congratulations on finishing this assignment! You are now able to implement an attention model and use it to learn complex mappings from one sequence to another. 